In [1]:
import sys
import json
sys.path.append("..")

from datasets import load_dataset
from src.utils import FIELDS, parse_ground_truth, get_field, normalize_number
from src.ocr import group_into_lines
from src.llm_extract import extract_fields

val = load_dataset("naver-clova-ix/cord-v2", cache_dir="../data", split="validation")
val_gt = list(val["ground_truth"])

with open("../results/ocr/validation.json", encoding="utf-8") as f:
    ocr = json.load(f)

In [2]:
for record in ocr["records"][:5]:
    lines = group_into_lines(record["words"])
    pred, stats = extract_fields(lines)
    gt = parse_ground_truth(val_gt[record["index"]])

    print(f"--- Zettel {record['index']}  ({stats['seconds']:.1f} s, {stats['prompt_tokens']} Prompt-Tokens)")
    for section, key in FIELDS:
        g = normalize_number(get_field(gt, section, key))
        p = normalize_number(pred.get(key))
        mark = "✓" if g == p else "✗"
        print(f"  {mark} {key:15s} gt={g}  pred={p}")

--- Zettel 0  (16.5 s, 319 Prompt-Tokens)
  ✓ total_price     gt=45500  pred=45500
  ✓ subtotal_price  gt=None  pred=None
  ✗ cashprice       gt=50000  pred=500000
  ✓ changeprice     gt=4500  pred=4500
  ✓ tax_price       gt=None  pred=None
--- Zettel 1  (6.8 s, 301 Prompt-Tokens)
  ✓ total_price     gt=23000  pred=23000
  ✓ subtotal_price  gt=None  pred=None
  ✗ cashprice       gt=50000  pred=5000
  ✗ changeprice     gt=27000  pred=2700
  ✓ tax_price       gt=None  pred=None
--- Zettel 2  (8.8 s, 339 Prompt-Tokens)
  ✗ total_price     gt=20000  pred=100000
  ✗ subtotal_price  gt=18181  pred=20000
  ✗ cashprice       gt=100000  pred=8000
  ✗ changeprice     gt=80000  pred=8000
  ✗ tax_price       gt=1818  pred=18181
--- Zettel 3  (7.2 s, 331 Prompt-Tokens)
  ✓ total_price     gt=28000  pred=28000
  ✓ subtotal_price  gt=28000  pred=28000
  ✓ cashprice       gt=28000  pred=28000
  ✗ changeprice     gt=0  pred=None
  ✓ tax_price       gt=0  pred=0
--- Zettel 4  (7.0 s, 311 Prompt-Tokens)